# FYS4480/9480, week 45: diagram rules, third order and the linked-diagram theorem

Companion notebook to the week-45 slides (November 5--6, 2026) and to chapter 9, section 9.15 of the lecture notes. Everything is self-contained numpy; the same code is `BookPrograms/chapter09/diagrams.py`.

**Contents**

1. The lists of week 44, lecture 4: the recursion against the explicit expressions through fourth order
2. Normal ordering absorbs the first-order renormalisation terms: $E^{(3)}=\langle W R_0 W R_0 W\rangle$
3. The two second-order diagrams, read with the rules
4. The fourteen third-order diagrams: random Hamiltonian, pairing model, pairing plus particle-hole model
5. Unlinked diagrams at fourth order: two non-interacting copies and the factorisation of the denominators
6. The week-44 exercise 2: two particles in a single $j$-shell (Brillouin-Wigner, diagonalisation, the fixed-point trap)

Notation: $H=H_0+H_I$, $H_0=\sum_p\varepsilon_p a^\dagger_p a_p$, $H_I=\sum_{pq}u_{pq}a^\dagger_pa_q+\frac14\sum\bar v_{pqrs}a^\dagger_pa^\dagger_qa_sa_r$; the reference fills the lowest $N$ orbitals; $R_0=Q/(\varepsilon_0-H_0)$; $W=H_I-E^{(1)}$ is the normal-ordered perturbation with the one-body vertex $\tilde f_{pq}=u_{pq}+U_{pq}$, $U_{pq}=\sum_{j\,\mathrm{occ}}\bar v_{pjqj}$.

## 0. The machinery

Bit-string determinants, a general two-body Hamiltonian in the full determinant space (the orbital-level data $\varepsilon$, $u$, $\bar v$ are kept, because the diagrams are evaluated from them), the Rayleigh-Schrödinger recursion of week 44, and the pieces of the third order.

In [1]:
import numpy as np
from itertools import combinations

import numpy as np


# ---------------------------------------------------------------------------
#  bit-string determinants
# ---------------------------------------------------------------------------
def popcount(x):
    return bin(x).count("1")


def annihilate(state, p):
    if not (state >> p) & 1:
        return 0, 0
    sign = -1 if popcount(state & ((1 << p) - 1)) & 1 else 1
    return sign, state ^ (1 << p)


def create(state, p):
    if (state >> p) & 1:
        return 0, 0
    sign = -1 if popcount(state & ((1 << p) - 1)) & 1 else 1
    return sign, state | (1 << p)


def apply_string(state, operators):
    """Apply [(index, dagger), ...], rightmost operator first."""
    sign = 1
    for p, dagger in reversed(operators):
        s, state = (create(state, p) if dagger else annihilate(state, p))
        if s == 0:
            return 0, 0
        sign *= s
    return sign, state


class GeneralHamiltonian:
    """H = sum_p eps_p a+_p a_p + sum_pq u_pq a+_p a_q + 1/4 sum vbar a+a+aa.

    The reference determinant fills the first n_particles orbitals; the
    orbital energies eps must therefore be ordered so that these are the
    lowest.  H_0 = sum eps_p a+_p a_p, H_I = the rest.
    """

    def __init__(self, eps, u, vbar, n_particles):
        self.eps = np.asarray(eps, dtype=float)
        self.u = np.asarray(u, dtype=float)
        self.vbar = np.asarray(vbar, dtype=float)
        self.n = len(self.eps)
        self.N = n_particles
        self.occ = list(range(n_particles))
        self.vir = list(range(n_particles, self.n))
        self.states = []
        for occ in combinations(range(self.n), n_particles):
            bits = 0
            for p in occ:
                bits |= 1 << p
            self.states.append(bits)
        self.states.sort()
        self.index = {s: i for i, s in enumerate(self.states)}
        self.dim = len(self.states)
        self.reference = (1 << n_particles) - 1
        self.ref = self.index[self.reference]
        self._build()

    # ------------------------------------------------------------------
    def _build(self):
        n, dim = self.n, self.dim
        H0 = np.zeros(dim)
        V = np.zeros((dim, dim))
        for col, state in enumerate(self.states):
            H0[col] = sum(self.eps[p] for p in range(n) if (state >> p) & 1)
            for p in range(n):
                for q in range(n):
                    if self.u[p, q] == 0.0:
                        continue
                    sign, new = apply_string(state, [(p, True), (q, False)])
                    if sign:
                        V[self.index[new], col] += sign * self.u[p, q]
            for p in range(n):
                for q in range(n):
                    for r in range(n):
                        for s in range(n):
                            v = self.vbar[p, q, r, s]
                            if v == 0.0:
                                continue
                            sign, new = apply_string(
                                state, [(p, True), (q, True), (s, False), (r, False)])
                            if sign:
                                V[self.index[new], col] += 0.25 * sign * v
        self.w = H0
        self.V = V
        self.H = np.diag(H0) + V
        self.w0 = H0[self.ref]
        D = self.w0 - H0
        self.R = np.zeros(dim)
        mask = np.arange(dim) != self.ref
        self.R[mask] = 1.0 / D[mask]
        self.D = D

    # ------------------------------------------------------------------
    def rayleigh_schrodinger(self, order=4):
        """Delta E^(1) ... Delta E^(order) by the standard recursion."""
        V, R, ref = self.V, self.R, self.ref
        psi0 = np.zeros(self.dim)
        psi0[ref] = 1.0
        energies = [float(psi0 @ (V @ psi0))]
        psis = []
        for k in range(1, order):
            previous = psis[-1] if psis else psi0
            rhs = V @ previous
            for m in range(1, k):
                rhs = rhs - energies[m - 1] * psis[k - m - 1]
            psi = R * rhs
            psis.append(psi)
            energies.append(float(psi0 @ (V @ psi)))
        return energies

    def third_order_pieces(self):
        """Principal term, renormalisation term, and <W R W R W> with the
        normal-ordered vertex W = V - V_00."""
        V, R, ref = self.V, self.R, self.ref
        v0 = V[:, ref]
        principal = float(v0 @ (R * (V @ (R * v0))))
        renorm = float(V[ref, ref] * np.sum(R * R * v0 * v0))
        W = V - V[ref, ref] * np.eye(self.dim)
        w0 = W[:, ref]
        normal = float(w0 @ (R * (W @ (R * w0))))
        return principal, renorm, normal

    def exact(self):
        return float(np.linalg.eigvalsh(self.H)[0])

    # ------------------------------------------------------------------
    def ftilde(self):
        """The normal-ordered one-body vertex u + U, U_pq = sum_j vbar_pjqj."""
        U = sum(self.vbar[:, j, :, j] for j in self.occ)
        return self.u + U




## 1. The lists of week 44, lecture 4

The recursion
$$|\Psi^{(n)}\rangle=R_0\Bigl[H_I|\Psi^{(n-1)}\rangle-\sum_{k=1}^{n-1}E^{(k)}|\Psi^{(n-k)}\rangle\Bigr],\qquad E^{(n+1)}=\langle\Phi_0|H_I|\Psi^{(n)}\rangle,$$
against the explicit fourth order
$$E^{(4)}=\langle VRVRVRV\rangle-E^{(1)}\bigl[\langle VR^2VRV\rangle+\langle VRVR^2V\rangle\bigr]+(E^{(1)})^2\langle VR^3V\rangle-E^{(2)}\langle VR^2V\rangle .$$

In [2]:
def random_hamiltonian(n=8, n_particles=4, seed=1, scale_u=0.1, scale_v=0.2):
    """A random Hermitian two-body Hamiltonian with every diagram class
    non-zero: eps increasing, u symmetric and off-diagonal, vbar
    antisymmetrised and Hermitian."""
    rng = np.random.default_rng(seed)
    eps = np.sort(rng.uniform(0.0, 1.0, n))
    eps = eps - eps[0] + np.linspace(0, 2, n)             # well separated
    u = scale_u * rng.normal(size=(n, n))
    u = 0.5 * (u + u.T)
    np.fill_diagonal(u, 0.0)
    w = scale_v * rng.normal(size=(n, n, n, n))
    # Hermitian: <pq|v|rs> = <rs|v|pq>
    w = 0.5 * (w + w.transpose(2, 3, 0, 1))
    # antisymmetrise in (p,q) and (r,s)
    vbar = w - w.transpose(1, 0, 2, 3)
    vbar = vbar - vbar.transpose(0, 1, 3, 2)
    vbar *= 0.5
    return GeneralHamiltonian(eps, u, vbar, n_particles)


def add_two_body(vbar, i, j, k, l, c):
    """Add c * a+_i a+_j a_k a_l to the interaction 1/4 sum vbar_pqrs a+_p a+_q a_s a_r,
    i.e. vbar_{ijlk} += c with its three antisymmetric partners."""
    vbar[i, j, l, k] += c
    vbar[j, i, l, k] -= c
    vbar[i, j, k, l] -= c
    vbar[j, i, k, l] += c


def pairing_hamiltonian(levels=4, particles=4, g=1.0, xi=1.0):
    """The pairing model: spin-orbital 2p = (p,+), 2p+1 = (p,-), levels
    p = 0..levels-1 with energy xi*p, interaction -g/2 sum_pq a+_{p+} a+_{p-} a_{q-} a_{q+}."""
    n = 2 * levels
    eps = np.array([xi * (o // 2) for o in range(n)], dtype=float)
    u = np.zeros((n, n))
    vbar = np.zeros((n, n, n, n))
    for p in range(levels):
        for q in range(levels):
            add_two_body(vbar, 2 * p, 2 * p + 1, 2 * q + 1, 2 * q, -0.5 * g)
    return GeneralHamiltonian(eps, u, vbar, particles)


def pairing_ph_hamiltonian(levels=4, particles=4, g=1.0, f=0.0, xi=1.0):
    """The pairing plus particle-hole model of chapter 7 (rpa.py):
    pairing as above plus
        -f/2 sum_{pqr} a+_{p+} a+_{p-} a_{q-} a_{r+}  + h.c."""
    ham = pairing_hamiltonian(levels, particles, g, xi)
    vbar = ham.vbar.copy()
    for p in range(levels):
        for q in range(levels):
            for r in range(levels):
                add_two_body(vbar, 2 * p, 2 * p + 1, 2 * q + 1, 2 * r, -0.5 * f)
                add_two_body(vbar, 2 * r, 2 * q + 1, 2 * p + 1, 2 * p, -0.5 * f)
    return GeneralHamiltonian(ham.eps, ham.u, vbar, particles)


def two_copies(ham_builder, **kwargs):
    """Two non-interacting copies of a Hamiltonian, as one Hamiltonian."""
    one = ham_builder(**kwargs)
    n, N = one.n, one.N
    # orbitals of copy 2 are shifted by n; the reference must fill the lowest
    # 2N orbitals, so interleave: copy-1 occupied, copy-2 occupied, then the rest
    order = list(range(N)) + list(range(n, n + N)) + list(range(N, n)) + list(range(n + N, 2 * n))
    eps2 = np.concatenate([one.eps, one.eps])[order]
    u2 = np.zeros((2 * n, 2 * n)); v2 = np.zeros((2 * n,) * 4)
    u_full = np.zeros((2 * n, 2 * n)); u_full[:n, :n] = one.u; u_full[n:, n:] = one.u
    v_full = np.zeros((2 * n,) * 4); v_full[:n, :n, :n, :n] = one.vbar; v_full[n:, n:, n:, n:] = one.vbar
    u2 = u_full[np.ix_(order, order)]
    v2 = v_full[np.ix_(order, order, order, order)]
    return one, GeneralHamiltonian(eps2, u2, v2, 2 * N)



def fourth_order_pieces(ham):
    """E4 = principal - E1[mixed] + E1^2 cubed - E2 folded, in resolvent form."""
    V, R, ref = ham.V, ham.R, ham.ref
    v0 = V[:, ref]
    e1 = V[ref, ref]
    e2 = float(np.sum(R * v0 * v0))
    r1 = R * v0                       # R V |0>
    principal = float(v0 @ (R * (V @ (R * (V @ r1)))))
    mixed = float(v0 @ (R * R * (V @ r1))) + float(v0 @ (R * (V @ (R * r1))))
    cubed = float(np.sum(R ** 3 * v0 * v0))
    folded = float(np.sum(R * R * v0 * v0))
    return principal - e1 * mixed + e1 ** 2 * cubed - e2 * folded, principal, mixed, cubed, folded


In [3]:
for name, ham in [("pairing g=0.5", pairing_hamiltonian(g=0.5)),
                  ("pairing+ph g=0.5 f=0.025", pairing_ph_hamiltonian(g=0.5, f=0.025))]:
    e = ham.rayleigh_schrodinger(4)
    e4, principal, mixed, cubed, folded = fourth_order_pieces(ham)
    e1, e2 = e[0], e[1]
    print(f"{name:28s} dim {ham.dim}")
    print(f"   E1..E4 (recursion) : {e[0]:.8f} {e[1]:.8f} {e[2]:.8f} {e[3]:.8f}")
    print(f"   E4 (explicit)      : {e4:.8f} = principal {principal:.8f} - E1*mixed {e1*mixed:.8f}"
          f" + E1^2*cubed {e1**2*cubed:.8f} - E2*folded {e2*folded:.8f}")
    print(f"   exact ground state : {ham.exact():.8f}")

pairing g=0.5                dim 70
   E1..E4 (recursion) : -0.50000000 -0.07291667 -0.01041667 -0.00034813
   E4 (explicit)      : -0.00034813 = principal -0.00816967 - E1*mixed -0.00849971 + E1^2*cubed -0.00251374 - E2*folded -0.00183558
   exact ground state : 1.41677428
pairing+ph g=0.5 f=0.025     dim 70
   E1..E4 (recursion) : -0.55000000 -0.08929167 -0.01386153 -0.00046834
   E4 (explicit)      : -0.00046834 = principal -0.01210186 - E1*mixed -0.01267086 + E1^2*cubed -0.00381055 - E2*folded -0.00277322
   exact ground state : 1.34713265


## 2. Normal ordering absorbs the first-order renormalisation terms

With $W=H_I-E^{(1)}$ the recursion has no $E^{(1)}$ term left: $E^{(3)}=\langle WR_0WR_0W\rangle$ with nothing to subtract, because in the determinant basis $W_{mn}=V_{mn}-V_{00}\delta_{mn}$ and the $\delta_{mn}$ part of $\sum V_{0m}W_{mn}V_{n0}/D_mD_n$ is exactly the renormalisation term $-V_{00}\sum|V_{0m}|^2/D_m^2$.

In [4]:
for name, ham in [("pairing g=0.5", pairing_hamiltonian(g=0.5)),
                  ("random n=8 N=4", random_hamiltonian())]:
    principal, renorm, normal = ham.third_order_pieces()
    e3 = ham.rayleigh_schrodinger(3)[2]
    print(f"{name:18s} principal {principal:14.10f}  renormalisation {renorm:14.10f}  "
          f"difference {principal - renorm:14.10f}  <WRWRW> {normal:14.10f}  E3 {e3:14.10f}")

pairing g=0.5      principal  -0.0230034722  renormalisation  -0.0125868056  difference  -0.0104166667  <WRWRW>  -0.0104166667  E3  -0.0104166667
random n=8 N=4     principal  -0.1417616190  renormalisation  -0.5414335818  difference   0.3996719628  <WRWRW>   0.3996719628  E3   0.3996719628


## 3. The second-order diagrams

Two diagrams with normal-ordered vertices: two crosses joined by a particle and a hole line ($h=l=1$, one cut), and the ring ($h=l=2$, two pairs of equivalent lines, four lines crossing the cut):
$$E^{(2)}=\sum_{ia}\frac{\tilde f_{ia}\tilde f_{ai}}{\varepsilon_i-\varepsilon_a}+\frac14\sum_{ijab}\frac{|\bar v_{abij}|^2}{\varepsilon_i+\varepsilon_j-\varepsilon_a-\varepsilon_b}.$$

In [5]:
# ---------------------------------------------------------------------------
#  the diagrams
# ---------------------------------------------------------------------------
def second_order_diagrams(ham):
    """The two second-order diagrams with normal-ordered vertices."""
    eps, vb = ham.eps, ham.vbar
    o, v = ham.occ, ham.vir
    f = ham.ftilde()
    e_o, e_v = eps[o], eps[v]
    Dai = e_o[:, None] - e_v[None, :]                       # [i, a]
    fov = f[np.ix_(o, v)]                                  # f_ia
    fvo = f[np.ix_(v, o)]                                  # f_ai
    singles = float(np.sum(fov * fvo.T / Dai))
    Dijab = (e_o[:, None, None, None] + e_o[None, :, None, None]
             - e_v[None, None, :, None] - e_v[None, None, None, :])
    v_oovv = vb[np.ix_(o, o, v, v)]
    v_vvoo = vb[np.ix_(v, v, o, o)]
    doubles = 0.25 * float(np.einsum("ijab,abij,ijab->", v_oovv, v_vvoo, 1.0 / Dijab))
    return {"(F,F) singles": singles, "(V,V) doubles": doubles}


def third_order_diagrams(ham):
    """The fourteen connected third-order energy diagrams.

    Vertices are read from the bottom (first to act) to the top, F is the
    normal-ordered one-body vertex ftilde and V the interaction.  Each value
    is obtained with the rules: sums over labels, one denominator per cut,
    1/2 per pair of equivalent lines, sign (-1)^(h+l).
    """
    eps, vb = ham.eps, ham.vbar
    o, v = ham.occ, ham.vir
    f = ham.ftilde()
    e_o, e_v = eps[o], eps[v]
    foo = f[np.ix_(o, o)]       # f_ij
    fvv = f[np.ix_(v, v)]       # f_ab
    fov = f[np.ix_(o, v)]       # f_ia
    fvo = f[np.ix_(v, o)]       # f_ai
    D1 = e_o[:, None] - e_v[None, :]                               # D[i,a]
    D2 = (e_o[:, None, None, None] + e_o[None, :, None, None]
          - e_v[None, None, :, None] - e_v[None, None, None, :])   # D[i,j,a,b]
    v_oovv = vb[np.ix_(o, o, v, v)]   # <ij|v|ab>
    v_vvoo = vb[np.ix_(v, v, o, o)]   # <ab|v|ij>
    v_ovov = vb[np.ix_(o, v, o, v)]   # <ib|v|ja>
    v_vvvo = vb[np.ix_(v, v, v, o)]   # <bc|v|aj>
    v_ovoo = vb[np.ix_(o, v, o, o)]   # <ib|v|jk>
    v_vvvv = vb[np.ix_(v, v, v, v)]   # <cd|v|ab>
    v_oooo = vb[np.ix_(o, o, o, o)]   # <ij|v|kl>
    v_vovo = vb[np.ix_(v, o, v, o)]   # <ci|v|ak>
    v_ovvv = vb[np.ix_(o, v, v, v)]   # <ic|v|ab>
    v_oovo = vb[np.ix_(o, o, v, o)]   # <ij|v|ak>

    d = {}
    # (F,F,F)
    d["1  (F,F,F) particle insertion"] = float(np.einsum(
        "ib,ba,ai,ia,ib->", fov, fvv, fvo, 1 / D1, 1 / D1))
    d["2  (F,F,F) hole insertion"] = -float(np.einsum(
        "ja,ij,ai,ia,ja->", fov, foo, fvo, 1 / D1, 1 / D1))
    # (F,F,V)
    d["3  (F,F,V) two crosses closed by V"] = float(np.einsum(
        "ijab,bj,ai,ia,ijab->", v_oovv, fvo, fvo, 1 / D1, 1 / D2))
    # (F,V,F)
    d["4  (F,V,F) particle-hole scattering"] = -float(np.einsum(
        "jb,ibja,ai,ia,jb->", fov, v_ovov, fvo, 1 / D1, 1 / D1))
    # (F,V,V)
    d["5  (F,V,V) particle line into V"] = 0.5 * float(np.einsum(
        "ijbc,bcaj,ai,ia,ijbc->", v_oovv, v_vvvo, fvo, 1 / D1, 1 / D2))
    d["6  (F,V,V) hole line into V"] = -0.5 * float(np.einsum(
        "jkab,ibjk,ai,ia,jkab->", v_oovv, v_ovoo, fvo, 1 / D1, 1 / D2))
    # (V,F,V)
    d["7  (V,F,V) particle insertion"] = 0.5 * float(np.einsum(
        "ijcb,ca,abij,ijab,ijcb->", v_oovv, fvv, v_vvoo, 1 / D2, 1 / D2))
    d["8  (V,F,V) hole insertion"] = -0.5 * float(np.einsum(
        "kjab,ik,abij,ijab,kjab->", v_oovv, foo, v_vvoo, 1 / D2, 1 / D2))
    # (V,F,F)
    d["9  (V,F,F) V closed by two crosses"] = float(np.einsum(
        "jb,ia,abij,ijab,jb->", fov, fov, v_vvoo, 1 / D2, 1 / D1))
    # (V,V,V)
    d["10 (V,V,V) particle-particle ladder"] = 0.125 * float(np.einsum(
        "ijcd,cdab,abij,ijab,ijcd->", v_oovv, v_vvvv, v_vvoo, 1 / D2, 1 / D2))
    d["11 (V,V,V) hole-hole ladder"] = 0.125 * float(np.einsum(
        "klab,ijkl,abij,ijab,klab->", v_oovv, v_oooo, v_vvoo, 1 / D2, 1 / D2))
    d["12 (V,V,V) particle-hole ring"] = -float(np.einsum(
        "kjcb,ciak,abij,ijab,kjcb->", v_oovv, v_vovo, v_vvoo, 1 / D2, 1 / D2))
    # (V,V,F)
    d["13 (V,V,F) V to single via particles"] = 0.5 * float(np.einsum(
        "jc,icab,abij,ijab,jc->", fov, v_ovvv, v_vvoo, 1 / D2, 1 / D1))
    d["14 (V,V,F) V to single via holes"] = -0.5 * float(np.einsum(
        "kb,ijak,abij,ijab,kb->", fov, v_oovo, v_vvoo, 1 / D2, 1 / D1))
    return d




In [6]:
for name, ham in [("random n=8 N=4", random_hamiltonian()), ("pairing g=0.5", pairing_hamiltonian(g=0.5)),
                  ("pairing+ph g=0.5 f=0.025", pairing_ph_hamiltonian(g=0.5, f=0.025))]:
    d2 = second_order_diagrams(ham); e2 = ham.rayleigh_schrodinger(2)[1]
    print(f"{name:26s}", {k: round(v, 10) for k, v in d2.items()}, " sum", round(sum(d2.values()), 10), " E2", round(e2, 10))

random n=8 N=4             {'(F,F) singles': -0.6507491772, '(V,V) doubles': -0.146956561}  sum -0.7977057382  E2 -0.7977057382
pairing g=0.5              {'(F,F) singles': 0.0, '(V,V) doubles': -0.0729166667}  sum -0.0729166667  E2 -0.0729166667
pairing+ph g=0.5 f=0.025   {'(F,F) singles': -0.0007291667, '(V,V) doubles': -0.0885625}  sum -0.0892916667  E2 -0.0892916667


## 4. The fourteen third-order diagrams

Each diagram is evaluated with the four rules -- sums over labels, one denominator per cut, $\tfrac12$ per pair of equivalent lines, the sign $(-1)^{h+l}$ -- from the orbital-level matrix elements, and their sum is compared with $E^{(3)}$ from the recursion in the full determinant space. The random Hamiltonian has every diagram non-zero; a wrong sign or weight in any one of them shows up in the last line.

In [7]:
def report(ham, title):
    d = third_order_diagrams(ham); e3 = ham.rayleigh_schrodinger(3)[2]
    print("=" * 72); print(title, " dim", ham.dim)
    for k, v in d.items():
        print(f"   {k:44s} {v:16.12f}")
    print(f"   {'sum of the fourteen':44s} {sum(d.values()):16.12f}")
    print(f"   {'E3, determinant space':44s} {e3:16.12f}")
    return d

d_random = report(random_hamiltonian(), "random n=8, N=4")

random n=8, N=4  dim 70
   1  (F,F,F) particle insertion                  0.029978686747
   2  (F,F,F) hole insertion                      0.222374015839
   3  (F,F,V) two crosses closed by V             0.064812429515
   4  (F,V,F) particle-hole scattering            0.152115742506
   5  (F,V,V) particle line into V               -0.023773776192
   6  (F,V,V) hole line into V                   -0.061323557991
   7  (V,F,V) particle insertion                  0.004225746070
   8  (V,F,V) hole insertion                      0.015920047350
   9  (V,F,F) V closed by two crosses             0.064812429515
   10 (V,V,V) particle-particle ladder            0.004426951312
   11 (V,V,V) hole-hole ladder                    0.002250486747
   12 (V,V,V) particle-hole ring                  0.008950095582
   13 (V,V,F) V to single via particles          -0.023773776192
   14 (V,V,F) V to single via holes              -0.061323557991
   sum of the fourteen                            0.399671962817
 

In [8]:
from fractions import Fraction
d_pair = report(pairing_hamiltonian(g=0.5), "pairing model, four levels, two pairs, g = 0.5")
print("\nin units of g^3 (g = 0.5 -> divide by 1/8):")
for k, v in d_pair.items():
    if abs(v) > 1e-12:
        print(f"   {k:44s} {Fraction(v * 8).limit_denominator(5000)}")
print("   sum", Fraction(sum(d_pair.values()) * 8).limit_denominator(5000))

pairing model, four levels, two pairs, g = 0.5  dim 70
   1  (F,F,F) particle insertion                  0.000000000000
   2  (F,F,F) hole insertion                     -0.000000000000
   3  (F,F,V) two crosses closed by V             0.000000000000
   4  (F,V,F) particle-hole scattering           -0.000000000000
   5  (F,V,V) particle line into V                0.000000000000
   6  (F,V,V) hole line into V                   -0.000000000000
   7  (V,F,V) particle insertion                  0.000000000000
   8  (V,F,V) hole insertion                      0.012586805556
   9  (V,F,F) V closed by two crosses             0.000000000000
   10 (V,V,V) particle-particle ladder           -0.011501736111
   11 (V,V,V) hole-hole ladder                   -0.011501736111
   12 (V,V,V) particle-hole ring                 -0.000000000000
   13 (V,V,F) V to single via particles           0.000000000000
   14 (V,V,F) V to single via holes              -0.000000000000
   sum of the fourteen             

Pairing model: only the two ladders and the hole-line insertion survive, $-\tfrac{53}{576}-\tfrac{53}{576}+\tfrac{29}{288}=-\tfrac1{12}$ in units of $g^3$. The ladders add up to the principal term of week 44 ($-0.02300347$) and the hole insertion is minus the renormalisation term ($+0.01258681$): for a pair excitation $W_{mm}=V_{mm}-V_{00}=0$, so the "renormalisation term" was the mean-field insertion $U_{ii}=-g/2$ on the two hole lines all along.

In [9]:
d_ph = report(pairing_ph_hamiltonian(g=0.5, f=0.025), "pairing + particle-hole, g = 0.5, f = 0.025")
# the switched-on diagrams are of order f^2 (times g or f): double f at fixed g
for f in (0.025, 0.05):
    d = third_order_diagrams(pairing_ph_hamiltonian(g=0.5, f=f))
    print(f"f = {f}: diagram 2 = {d['2  (F,F,F) hole insertion']:.3e}, diagram 3 = {d['3  (F,F,V) two crosses closed by V']:.3e}")

pairing + particle-hole, g = 0.5, f = 0.025  dim 70
   1  (F,F,F) particle insertion                  0.000000000000
   2  (F,F,F) hole insertion                      0.000148871528
   3  (F,F,V) two crosses closed by V            -0.000074435764
   4  (F,V,F) particle-hole scattering           -0.000000000000
   5  (F,V,V) particle line into V               -0.000131380208
   6  (F,V,V) hole line into V                    0.000131380208
   7  (V,F,V) particle insertion                  0.000000000000
   8  (V,F,V) hole insertion                      0.016919565972
   9  (V,F,F) V closed by two crosses            -0.000074435764
   10 (V,V,V) particle-particle ladder           -0.015390546875
   11 (V,V,V) hole-hole ladder                   -0.015390546875
   12 (V,V,V) particle-hole ring                 -0.000000000000
   13 (V,V,F) V to single via particles          -0.000131380208
   14 (V,V,F) V to single via holes               0.000131380208
   sum of the fourteen                

f = 0.025: diagram 2 = 1.489e-04, diagram 3 = -7.444e-05
f = 0.05: diagram 2 = 6.875e-04, diagram 3 = -3.438e-04


## 5. Unlinked diagrams at fourth order

Two non-interacting copies $A$ and $B$ of a three-level pairing system with one pair each (no shared orbitals, hence no exclusion-principle-violating terms). With normal-ordered vertices $E^{(4)}=\langle WR_0WR_0WR_0W\rangle-E^{(2)}\langle WR_0^2W\rangle$. Restricting the intermediate states of the principal term to the four patterns $(A,AB,B)$, $(B,AB,A)$, $(A,AB,A)$, $(B,AB,B)$ isolates the unlinked diagrams; by the factorisation of the denominators,
$$\frac{1}{D_m(D_m+D_n)D_n}+\frac{1}{D_m(D_m+D_n)D_m}=\frac{1}{D_m^2D_n},$$
their sum must equal the cross-copy part of the folded term, $2E^{(2)}_A\langle WR_0^2W\rangle_A$.

In [10]:
def fourth_order_unlinked(builder, **kwargs):
    """Two non-interacting copies A and B.  With normal-ordered vertices W,
    E4 = <WRWRWRW> - E2 <WR^2W>.  The part of the principal term in which the
    intermediate states visit A, then A and B together, then B (or A, AB, A,
    and the two with the roles exchanged) is the sum of the unlinked
    fourth-order diagrams; it must equal the cross-copy part of the
    renormalisation term, 2 E2(one copy) <WR^2W>(one copy)."""
    one, two = two_copies(builder, **kwargs)
    n, N = one.n, one.N
    # which copy is excited in each determinant of the two-copy system
    # orbitals 0..N-1: A occupied, N..2N-1: B occupied, 2N..N+n-1: A empty, rest B empty
    def excited(state):
        a = sum(1 for p in range(2 * N, N + n) if (state >> p) & 1)
        b = sum(1 for p in range(N + n, 2 * n) if (state >> p) & 1)
        return a > 0, b > 0
    flags = [excited(s) for s in two.states]
    PA = np.array([a and not b for a, b in flags], float)
    PB = np.array([b and not a for a, b in flags], float)
    PAB = np.array([a and b for a, b in flags], float)
    W = two.V - two.V[two.ref, two.ref] * np.eye(two.dim)
    R = two.R
    w0 = W[:, two.ref]
    def chain(P1, P2, P3):
        x = R * P1 * w0
        x = R * P2 * (W @ x)
        x = R * P3 * (W @ x)
        return float(w0 @ x)
    unlinked = chain(PA, PAB, PB) + chain(PB, PAB, PA) + chain(PA, PAB, PA) + chain(PB, PAB, PB)
    W1 = one.V - one.V[one.ref, one.ref] * np.eye(one.dim)
    w1 = W1[:, one.ref]
    e2_one = float(np.sum(one.R * w1 * w1))
    r2_one = float(np.sum(one.R ** 2 * w1 * w1))
    e_one = one.rayleigh_schrodinger(4)
    e_two = two.rayleigh_schrodinger(4)
    return {"unlinked diagrams (four time orderings)": unlinked,
            "2 E2(A) <WR^2W>(A)": 2 * e2_one * r2_one,
            "E4 two copies": e_two[3], "2 x E4 one copy": 2 * e_one[3],
            "A,AB,B": chain(PA, PAB, PB), "A,AB,A": chain(PA, PAB, PA)}




In [11]:
res = fourth_order_unlinked(pairing_hamiltonian, levels=3, particles=2, g=0.5)
for k, v in res.items():
    print(f"   {k:42s} {v:18.14f}")
print("   the (A,AB,B) and (A,AB,A) orderings alone are not products; only their sum factorises")

   unlinked diagrams (four time orderings)     -0.00183105468750
   2 E2(A) <WR^2W>(A)                          -0.00183105468750
   E4 two copies                                0.00109863281250
   2 x E4 one copy                              0.00109863281250
   A,AB,B                                      -0.00043741861979
   A,AB,A                                      -0.00047810872396
   the (A,AB,B) and (A,AB,A) orderings alone are not products; only their sum factorises


## 6. Week-44 exercise 2: two particles in a single $j$-shell

$\Omega=j+\tfrac12$ degenerate pair states, $H=-G\,J$ with $J$ the all-ones matrix. Brillouin-Wigner sums a geometric series to $E_0(E_0+\Omega G)=0$, i.e. $E_0=-\Omega G$; diagonalisation confirms it (rank-one $J$: eigenvalues $\Omega$ once and $0$ otherwise). Rayleigh-Schrödinger cannot start: every denominator $\varepsilon_0-\varepsilon_m$ vanishes. The naive fixed-point iteration of the BW equation converges to the *unphysical* root $0$.

In [12]:
def bisect(F, a, b, tol=1e-13):
    fa = F(a)
    for _ in range(200):
        m = 0.5 * (a + b); fm = F(m)
        if fa * fm <= 0: b = m
        else: a, fa = m, fm
        if b - a < tol: break
    return 0.5 * (a + b)

for j2 in (5, 7, 9):                       # 2j
    Omega, G = (j2 + 1) // 2, 1.0
    H = -G * np.ones((Omega, Omega))
    exact = np.linalg.eigvalsh(H)
    f = lambda E: -G + (Omega - 1) * G**2 / (E + (Omega - 1) * G)   # the BW equation E = f(E)
    F = lambda E: E - f(E)
    E = -Omega - 0.37
    for _ in range(200): E = f(E)                                   # plain fixed-point iteration
    print(f"Omega = {Omega}: eigenvalues {np.round(exact, 10)}, BW roots {bisect(F, -Omega-1.0, -Omega+0.4):.12f} "
          f"and {bisect(F, -0.31, 0.3):.12f}, fixed-point iteration ends at {E:.6f}")

Omega = 3: eigenvalues [-3.  0.  0.], BW roots -3.000000000000 and -0.000000000000, fixed-point iteration ends at 0.000000
Omega = 4: eigenvalues [-4. -0. -0.  0.], BW roots -4.000000000000 and -0.000000000000, fixed-point iteration ends at 0.000000
Omega = 5: eigenvalues [-5. -0. -0.  0.  0.], BW roots -5.000000000000 and -0.000000000000, fixed-point iteration ends at 0.000000
